# Text Data Augmentation and Classification with nlpaug

This notebook shows:
- text classification on a small IMDB subset
- text augmentation using `nlpaug`
- A/B test: baseline vs augmented data

In [ ]:
# Uncomment in Colab if needed:
# !pip -q install nlpaug

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
import nlpaug.augmenter.char as nac

max_words = 10000
max_len = 120

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_words)

word_index = keras.datasets.imdb.get_word_index()
reverse_index = {value + 3: key for key, value in word_index.items()}
reverse_index[0] = "<PAD>"
reverse_index[1] = "<START>"
reverse_index[2] = "<UNK>"
reverse_index[3] = "<UNUSED>"

def decode_review(encoded_review):
    return " ".join(reverse_index.get(i, "?") for i in encoded_review)

texts_train = [decode_review(x) for x in x_train[:2000]]
labels_train = np.array(y_train[:2000])
texts_test = [decode_review(x) for x in x_test[:1000]]
labels_test = np.array(y_test[:1000])

aug = nac.RandomCharAug(action="delete")

augmented_texts = []
augmented_labels = []
for text, label in zip(texts_train[:1000], labels_train[:1000]):
    augmented_texts.append(aug.augment(text))
    augmented_labels.append(label)

train_texts_base = texts_train
train_labels_base = labels_train

train_texts_aug = texts_train + augmented_texts
train_labels_aug = np.concatenate([labels_train, np.array(augmented_labels)])

In [ ]:
vectorizer = layers.TextVectorization(
    max_tokens=max_words,
    output_mode="int",
    output_sequence_length=max_len
)
vectorizer.adapt(train_texts_base)

def build_text_model():
    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = layers.Embedding(max_words, 64)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

base_model = build_text_model()
base_model.fit(np.array(train_texts_base), train_labels_base, epochs=3, batch_size=64, verbose=0)
base_acc = base_model.evaluate(np.array(texts_test), labels_test, verbose=0)[1]

aug_model = build_text_model()
aug_model.fit(np.array(train_texts_aug), train_labels_aug, epochs=3, batch_size=64, verbose=0)
aug_acc = aug_model.evaluate(np.array(texts_test), labels_test, verbose=0)[1]

print("Baseline accuracy:", round(base_acc, 4))
print("Augmented accuracy:", round(aug_acc, 4))